<a href="https://colab.research.google.com/github/Mansehaj-Singh/Drone-Orientation-Estimation/blob/main/Mansehaj_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import scipy.io as sio
import numpy as np
import os


# 1. CONFIGURATION

class Config:

    DATA_PATH = "fast_v4.mat"
    WINDOW_SIZE = 100          # 1 second of data at 100 Hz [cite: 2]
    INPUT_CHANNELS = 9         # Acc(3) + Gyro(3) + Mag(3) [cite: 8, 9, 10]
    NUM_LAYERS = 17            # Depth for Denoising CNN
    BATCH_SIZE = 32
    LR = 1e-4
    EPOCHS = 30
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# 2. DATASET LOADER

class DroneMATDataset(Dataset):
    def __init__(self, file_path, window_size=Config.WINDOW_SIZE):
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"CRITICAL: {file_path} not found. Upload it to Colab!")

        # Load the .mat file [cite: 2]
        mat_contents = sio.loadmat(file_path)

        # Automatically find the data matrix (skipping metadata) [cite: 6]
        data_keys = [k for k in mat_contents.keys() if not k.startswith('__')]
        if not data_keys:
            raise ValueError("No valid data arrays found in the .mat file.")

        # Select the first available sensor dataset (e.g., XS1 or AP1) [cite: 3, 4]
        raw_data = mat_contents[data_keys[0]]
        print(f"Successfully loaded sensor data: {data_keys[0]} with shape {raw_data.shape}")

        self.window_size = window_size

        # Features: Columns 2:10 (Acc, Gyro, Mag) [cite: 8, 9, 10]
        # Map indices 1 through 9
        self.features = raw_data[:, 1:10].astype(np.float32)

        # Target: Qs (q0, qx, qy, qz) [cite: 13]
        # Map indices 10 through 14 (Columns 11:14) [cite: 11]
        self.targets = raw_data[:, 10:14].astype(np.float32)

    def __len__(self):
        return len(self.features) - self.window_size

    def __getitem__(self, idx):
        x = self.features[idx : idx + self.window_size]
        y = self.targets[idx + self.window_size]
        return torch.tensor(x), torch.tensor(y)


# 3. DnCNN ARCHITECTURE (Model-Based)

class DnCNN_IMU(nn.Module):
    def __init__(self, channels=Config.INPUT_CHANNELS, num_layers=Config.NUM_LAYERS):
        super(DnCNN_IMU, self).__init__()
        layers = []
        # Initial Conv Layer
        layers.append(nn.Conv1d(channels, 64, kernel_size=3, padding=1))
        layers.append(nn.ReLU(inplace=True))

        # Deep Denoising Layers (Conv + BN + ReLU)
        for _ in range(num_layers - 2):
            layers.append(nn.Conv1d(64, 64, kernel_size=3, padding=1))
            layers.append(nn.BatchNorm1d(64))
            layers.append(nn.ReLU(inplace=True))

        # Final Reconstruction Layer
        layers.append(nn.Conv1d(64, channels, kernel_size=3, padding=1))
        self.dncnn = nn.Sequential(*layers)

    def forward(self, x):
        # x: (Batch, Seq, Feat) -> Transform for Conv1d: (Batch, Feat, Seq)
        x_trans = x.transpose(1, 2)

        # The DnCNN predicts the NOISE [Residual Learning]
        noise_estimate = self.dncnn(x_trans)
        denoised_signal = x_trans - noise_estimate

        return denoised_signal.transpose(1, 2)


# 4. EXECUTION BLOCK

def run_experiment():
    print(f"Running on: {Config.DEVICE}")

    # 1. Load Data
    try:
        dataset = DroneMATDataset(Config.DATA_PATH)
        loader = DataLoader(dataset, batch_size=Config.BATCH_SIZE, shuffle=True)
        print(f"Dataset size: {len(dataset)} samples")
    except Exception as e:
        print(f"Setup Error: {e}")
        return

    # 2. Initialize Model
    model = DnCNN_IMU().to(Config.DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=Config.LR)
    criterion = nn.MSELoss()

    # 3. Training Loop
    model.train()
    print("\n--- Starting Training ---")
    for epoch in range(Config.EPOCHS):
        epoch_loss = 0
        for x, y in loader:
            x, y = x.to(Config.DEVICE), y.to(Config.DEVICE)

            optimizer.zero_grad()
            out = model(x)

            # Use the denoised output to estimate orientation (first 4 channels)
            pred_q = out[:, -1, :4]
            loss = criterion(pred_q, y)

            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch [{epoch+1}/{Config.EPOCHS}] | Loss: {epoch_loss/len(loader):.6f}")

    # 4. Final RMSE Evaluation
    print("\n--- Final Evaluation ---")
    model.eval()
    eval_loader = DataLoader(dataset, batch_size=1, shuffle=False)
    all_preds, all_gts = [], []

    with torch.no_grad():
        for x, y in eval_loader:
            x = x.to(Config.DEVICE)
            out = model(x)
            # Normalize the predicted quaternion for valid orientation [cite: 13]
            pred_q = F.normalize(out[:, -1, :4], p=2, dim=1)

            all_preds.append(pred_q.cpu().numpy())
            all_gts.append(y.numpy())

    preds = np.vstack(all_preds)
    gts = np.vstack(all_gts)

    # Calculate RMSE per component and total [cite: 13]
    rmse_vec = np.sqrt(np.mean((preds - gts)**2, axis=0))
    total_rmse = np.sqrt(np.mean((preds - gts)**2))

    print("-" * 40)
    print(f"Component RMSE (q0, qx, qy, qz):")
    print(f" {rmse_vec}")
    print(f"Total Drone Orientation RMSE: {total_rmse:.6f}")
    print("-" * 40)

if __name__ == "__main__":
    run_experiment()

Running on: cpu
Successfully loaded sensor data: AP1 with shape (18995, 14)
Dataset size: 18895 samples

--- Starting Training ---
Epoch [1/30] | Loss: 3.383561
Epoch [5/30] | Loss: 0.231148
Epoch [10/30] | Loss: 0.146848
Epoch [15/30] | Loss: 0.095409
Epoch [20/30] | Loss: 0.075968
Epoch [25/30] | Loss: 0.067522
Epoch [30/30] | Loss: 0.053787

--- Final Evaluation ---
----------------------------------------
Component RMSE (q0, qx, qy, qz):
 [0.15426987 0.18894503 0.3087384  0.1350721 ]
Total Drone Orientation RMSE: 0.208005
----------------------------------------
